# M1 Chat SFT Data Generation (`m1_teacher_gen`) — P-1 Approved Mix + `numkeep.py`

- **Task (`AR-04`):** Generate the new chat SFT dataset from the frozen chat templates (`student_chat_template.txt` & `student_chat_template_hi.txt`) over non-excluded `train`/`dev` RHP passages (`[n] (RHP page p)`), filter all teacher outputs with `ml/handoff/eval/numkeep.py`, and mix with the existing risk-rewrite SFT data (`student_v1_train.jsonl.gz` 10,362 rows / `student_v1_dev.jsonl.gz` 529 rows).
- **Approved P-1 Chat SFT Mix (~3,000 chat rows total):**
  1. **1,500 scripted same-section hard near-miss unanswerables** (`label_source: "rule"`, EN 80% / HI 20%): multi-number passages from the right category where the requested metric/period/entity is verified absent $\to$ exact abstain string (`"I could not find this in the document."` / `"मुझे यह जानकारी दस्तावेज़ में नहीं मिली।"`).
  2. **~500 teacher-generated adversarial near-miss unanswerables** (`label_source: "ai"`): teacher crafts a plausible same-section question over `[n] (RHP page p)` blocks whose specific figure/period is not in the passages $\to$ exact abstain string.
  3. **~1,000 teacher-generated answerable numeric Q&A pairs** (`label_source: "ai"`): teacher writes a question and a 1–3 sentence answer copying numbers & units verbatim with `[n]` citations, filtered by `ml/handoff/eval/numkeep.py` (`finsight_lite.numcheck.unmatched_numbers`).
- **Expected Compute Units:**
  - **`DEBUG = True` (T4 smoke run):** ~4 min (~0.10 Colab units; generates 16 scripted + 12 teacher rows using `Qwen/Qwen3-4B-Instruct-2507` on T4, runs `numkeep.py`, and verifies the mixed SFT output).
  - **`DEBUG = False` (G4 full teacher run):** ~90 min (**13.07 G4 units** at 8.71 units/hr; checkpoints every chunk to `/content/drive/MyDrive/finsight_p4/m1_teacher_gen/epoch-<k>`).

In [ ]:
# First cell of every ARGON notebook (from ml/handoff/colab/setup_cell.py)
import json
import os
import pathlib
import random
import shutil
import subprocess
import sys

HF_REPO = "AkshatTm/finsight-gpu-jobs"  # private dataset repo (C-ADR-13)
HF_PREFIX = "phase4/handoff"  # folder in that repo; revision pinned in ml/handoff/README.md
HF_REVISION = "a285902d91a5557cf6d6d7cf23a9e266410d6c3d"  # upload_large revision in MANIFEST.json
HF_PACK_REVISION = "af028c61cde3f0aee6e52fc40593ac47b2a7aa1b"  # upload_pack commit in export_stats.json
JOB = "m1_teacher_gen"
DEBUG = True  # True = tiny slice for T4 debug run; set False on G4 ONLY after DEBUG passes
SEEDS = [13, 42, 2026]
RESUME = True  # continue from the newest checkpoint on Drive if one exists

ROOT = pathlib.Path("/content")
PACK = ROOT / "handoff"
DRIVE = pathlib.Path("/content/drive/MyDrive/finsight_p4")
CKPT_DIR = DRIVE / JOB


def mount_drive() -> None:
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive not mounted:", exc)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)


def hf_token() -> str:
    try:
        from google.colab import userdata  # type: ignore

        tok = userdata.get("HF_TOKEN")
        if tok:
            return tok.strip()
    except Exception:
        pass
    if "HF_TOKEN" in os.environ and os.environ["HF_TOKEN"].strip():
        return os.environ["HF_TOKEN"].strip()
    raise RuntimeError(
        "HF_TOKEN not found! In Colab, click the key icon (Secrets) on the left sidebar, "
        "add HF_TOKEN, and toggle 'Notebook access' ON."
    )


def fetch_pack() -> pathlib.Path:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import snapshot_download

    token = hf_token()
    hf_cache_dir = ROOT / "_hf"
    for rev in (HF_PACK_REVISION, None, HF_REVISION):
        try:
            snapshot_download(
                repo_id=HF_REPO,
                repo_type="dataset",
                revision=rev,
                token=token,
                allow_patterns=[f"{HF_PREFIX}/**", f"{HF_PREFIX}/*"],
                local_dir=str(hf_cache_dir),
            )
            if (hf_cache_dir / HF_PREFIX / "data" / "MANIFEST.json").exists():
                break
        except Exception as exc:
            print(f"snapshot_download (rev={rev}) note:", exc)

    for big_file in ("classifier_train.jsonl.gz", "risk_bank.parquet", "student_v1_train.jsonl.gz"):
        if not (hf_cache_dir / HF_PREFIX / "data" / big_file).exists():
            snapshot_download(
                repo_id=HF_REPO,
                repo_type="dataset",
                revision=HF_REVISION,
                token=token,
                allow_patterns=[f"{HF_PREFIX}/data/{big_file}"],
                local_dir=str(hf_cache_dir),
            )

    src = hf_cache_dir / HF_PREFIX
    assert (src / "data" / "MANIFEST.json").exists(), f"MANIFEST.json not found in {src}/data!"
    if PACK.exists():
        shutil.rmtree(PACK)
    shutil.copytree(src, PACK)
    return PACK


def verify_manifest() -> None:
    import hashlib

    manifest = json.loads((PACK / "data" / "MANIFEST.json").read_text(encoding="utf-8"))
    for f in manifest["files"]:
        h = hashlib.sha256((PACK / "data" / f["path"]).read_bytes()).hexdigest()
        assert h == f["sha256"], f"checksum mismatch: {f['path']}"


def install_deps() -> None:
    req_file = PACK / "colab" / "requirements_colab.txt"
    # On Colab, numpy/pandas/scipy are already imported in sys.modules by google.colab;
    # overwriting numpy/pandas in-place without restarting breaks C-extension ABI in Cell 3.
    in_colab = "google.colab" in sys.modules
    if in_colab and req_file.exists():
        lines = [
            ln.strip()
            for ln in req_file.read_text(encoding="utf-8").splitlines()
            if ln.strip() and not ln.strip().startswith("#")
        ]
        safe_pkgs = [
            ln for ln in lines if not ln.startswith(("numpy==", "pandas==", "pyarrow==", "scikit-learn=="))
        ]
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *safe_pkgs, "sentencepiece", "tiktoken"],
            check=False,
        )
    else:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file), "sentencepiece", "tiktoken"],
            check=True,
        )


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np

        np.random.seed(seed)
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def latest_checkpoint() -> pathlib.Path | None:
    ckpts = sorted(CKPT_DIR.glob("epoch-*"), key=lambda p: int(p.name.split("-")[1]))
    return ckpts[-1] if RESUME and ckpts else None


mount_drive()
fetch_pack()
verify_manifest()
install_deps()
sys.path.insert(0, str(PACK / "eval" / "vendor"))
print("pack ready:", PACK, "| DEBUG =", DEBUG, "| resume from:", latest_checkpoint())

In [ ]:
# Cell 2: Self-contained chat SFT builder (mirrors ml/m1_student_v2/chat_sft_builder.py) + 1,500 scripted near-miss unanswerables
import gzip
import re
import time
from collections import Counter, defaultdict
from typing import Any

from finsight_lite.numcheck import unmatched_numbers

t0_wall = time.time()
set_seed(SEEDS[0])

ABSTAIN_EN = "I could not find this in the document."
ABSTAIN_HI = "मुझे यह जानकारी दस्तावेज़ में नहीं मिली।"
_CITATION_RE = re.compile(r"\[\d+\]")
_NUMBER_HINT_RE = re.compile(r"(?:₹|rs\.?|inr|\d+(?:\.\d+)?\s*(?:%|million|crore|lakh))", re.I)

NEAR_MISS_TEMPLATES: dict[str, list[tuple[str, str, tuple[str, ...]]]] = {
    "financial": [
        (
            "What was the company's net cash flow from operating activities in the latest full financial year?",
            "नवीनतम पूर्ण वित्तीय वर्ष में कंपनी का परिचालन गतिविधियों से शुद्ध नकदी प्रवाह (net cash flow from operating activities) कितना था?",
            ("cash flow from operating", "operating activities", "परिचालन गतिविधियों"),
        ),
        (
            "What was the company's net worth at the end of the latest full financial year?",
            "नवीनतम पूर्ण वित्तीय वर्ष के अंत में कंपनी की नेट वर्थ (net worth) कितनी थी?",
            ("net worth", "नेट वर्थ"),
        ),
        (
            "What revenue from operations does the company forecast for the financial year 2030?",
            "वित्तीय वर्ष 2030 के लिए कंपनी परिचालन से कितने राजस्व (revenue from operations) का अनुमान लगाती है?",
            ("2030", "forecast for the financial year 2030"),
        ),
        (
            "What was the company's restated profit or loss after tax in Fiscal 2017?",
            "वित्तीय वर्ष 2017 में कर के बाद कंपनी का पुनर्गठित लाभ या हानि (restated profit or loss after tax) क्या था?",
            ("2017", "fiscal 2017", "fy17", "fy 2017"),
        ),
        (
            "What was the company's return on net worth (RoNW) in the latest full financial year?",
            "नवीनतम पूर्ण वित्तीय वर्ष में कंपनी का रिटर्न ऑन नेट वर्थ (RoNW) कितना था?",
            ("return on net worth", "ronw"),
        ),
    ],
    "debt_liquidity": [
        (
            "What was the exact interest rate per annum charged on the company's unsecured commercial paper in Fiscal 2024?",
            "वित्तीय वर्ष 2024 में कंपनी के असुरक्षित कमर्शियल पेपर (commercial paper) पर प्रति वर्ष सटीक ब्याज दर क्या थी?",
            ("commercial paper", "कमर्शियल पेपर"),
        ),
        (
            "What was the company's debt-to-equity ratio at the end of Fiscal 2018?",
            "वित्तीय वर्ष 2018 के अंत में कंपनी का ऋण-इक्विटी अनुपात (debt-to-equity ratio) क्या था?",
            ("2018", "debt-to-equity", "debt equity ratio"),
        ),
        (
            "How much money from the Fresh Issue is allocated specifically to redeeming preference shares?",
            "फ्रेश इश्यू (Fresh Issue) से कितनी राशि विशेष रूप से प्रेफरेंस शेयरों के मोचन (redemption of preference shares) के लिए आवंटित की गई है?",
            ("preference share", "redeeming preference", "redemption of preference"),
        ),
        (
            "What were the company's total borrowings at the end of Fiscal 2016?",
            "वित्तीय वर्ष 2016 के अंत में कंपनी का कुल उधार (total borrowings) कितना था?",
            ("2016", "fiscal 2016", "fy16"),
        ),
    ],
    "legal": [
        (
            "What was the total amount of compounding fees paid by the company to the Enforcement Directorate in Fiscal 2023?",
            "वित्तीय वर्ष 2023 में प्रवर्तन निदेशालय (Enforcement Directorate) को कंपनी द्वारा भुगतान किए गए कंपाउंडिंग शुल्क की कुल राशि क्या थी?",
            ("enforcement directorate", "pmla", "प्रवर्तन निदेशालय"),
        ),
        (
            "How many criminal proceedings are currently pending against the Independent Directors before the Supreme Court of India?",
            "भारत के सर्वोच्च न्यायालय के समक्ष स्वतंत्र निदेशकों (Independent Directors) के विरुद्ध वर्तमान में कितने आपराधिक मामले लंबित हैं?",
            ("supreme court", "independent director", "स्वतंत्र निदेशक"),
        ),
        (
            "What was the arbitral award amount granted against the company by the Singapore International Arbitration Centre?",
            "सिंगापुर इंटरनेशनल आर्बिट्रेशन सेंटर द्वारा कंपनी के विरुद्ध पारित आर्बिट्रल अवार्ड की राशि क्या थी?",
            ("singapore", "siac", "arbitral award"),
        ),
    ],
    "regulatory": [
        (
            "What monetary penalty was imposed on the company by the Competition Commission of India (CCI)?",
            "भारतीय प्रतिस्पर्धा आयोग (CCI) द्वारा कंपनी पर कितना मौद्रिक जुर्माना लगाया गया था?",
            ("competition commission", "cci", "प्रतिस्पर्धा आयोग"),
        ),
        (
            "How much customs duty penalty did the Directorate of Revenue Intelligence levy on the company in Fiscal 2022?",
            "वित्तीय वर्ष 2022 में राजस्व खुफिया निदेशालय (DRI) ने कंपनी पर कितना सीमा शुल्क जुर्माना लगाया?",
            ("directorate of revenue intelligence", "dri", "customs duty penalty"),
        ),
    ],
    "governance": [
        (
            "What is the exact post-Offer shareholding percentage of the Promoter Group after the exercise of the green shoe option?",
            "ग्रीन शू विकल्प (green shoe option) के प्रयोग के बाद प्रमोटर समूह की सटीक पोस्ट-ऑफर शेयरधारिता प्रतिशत क्या है?",
            ("green shoe", "post-offer shareholding", "ग्रीन शू"),
        ),
        (
            "What was the total sitting fee paid to the Chairman of the Audit Committee in Fiscal 2023?",
            "वित्तीय वर्ष 2023 में ऑडिट समिति के अध्यक्ष को भुगतान किया गया कुल सिटिंग शुल्क (sitting fee) कितना था?",
            ("sitting fee", "audit committee", "सिटिंग शुल्क"),
        ),
    ],
    "default": [
        (
            "How much money is the fresh issue raising?",
            "फ्रेश इश्यू (fresh issue) के माध्यम से कितनी राशि जुटाई जा रही है?",
            ("fresh issue", "offer for sale", "net proceeds"),
        ),
        (
            "What is the offer price per equity share?",
            "प्रति इक्विटी शेयर ऑफर मूल्य (offer price) क्या है?",
            ("offer price", "price band", "floor price", "cap price"),
        ),
        (
            "What is the size of the offer for sale?",
            "ऑफर फॉर सेल (offer for sale) का आकार क्या है?",
            ("offer for sale", "selling shareholder", "fresh issue"),
        ),
        (
            "What was the company's share price at the close of its first day of trading on the stock exchange?",
            "स्टॉक एक्सचेंज में कारोबार के पहले दिन के अंत में कंपनी का शेयर मूल्य क्या था?",
            ("first day of trading", "closing price on listing"),
        ),
        (
            "What was the grey market premium for this IPO on the day before it opened?",
            "इस IPO के खुलने से एक दिन पहले इसका ग्रे मार्केट प्रीमियम (grey market premium) क्या था?",
            ("grey market", "gmp"),
        ),
    ],
}


def format_passage_blocks(passages: list[dict[str, Any]]) -> str:
    blocks: list[str] = []
    for idx, p in enumerate(passages, start=1):
        doc_type = str(p.get("doc_type") or "RHP").upper()
        page = p.get("page") or p.get("page_start") or 1
        text = " ".join(str(p.get("text") or "").split())
        blocks.append(f"[{idx}] ({doc_type} page {page})\n{text}")
    return "\n\n".join(blocks)


def render_chat_prompt(question: str, passages: list[dict[str, Any]], lang: str = "en") -> str:
    fname = "student_chat_template_hi.txt" if lang == "hi" else "student_chat_template.txt"
    tpl = (PACK / "data" / fname).read_text(encoding="utf-8")
    blocks_str = format_passage_blocks(passages)
    return tpl.replace("{{QUESTION}}", question.strip()).replace("{{PASSAGE_BLOCKS}}", blocks_str).strip()


def build_scripted_near_miss_unanswerables(
    risk_rows: list[dict[str, Any]],
    excluded_ipos: set[str],
    target_count: int = 1500,
    seed: int = 42,
    hi_ratio: float = 0.20,
) -> list[dict[str, Any]]:
    rng = random.Random(seed)
    by_ipo_cat: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)
    for r in risk_rows:
        ipo_id = str(r.get("ipo_id") or "").strip()
        if not ipo_id or ipo_id in excluded_ipos or r.get("excluded"):
            continue
        split = str(r.get("split") or r.get("slice") or "train").strip()
        if split not in ("train", "dev"):
            continue
        text = " ".join(str(r.get("text") or "").split())
        if len(text.split()) < 25 or not _NUMBER_HINT_RE.search(text):
            continue
        cat = str(r.get("label") or r.get("category") or "default").strip()
        by_ipo_cat[(ipo_id, cat)].append(
            {
                "risk_id": r.get("risk_id") or r.get("id") or f"{ipo_id}#p",
                "ipo_id": ipo_id,
                "company": r.get("company") or ipo_id,
                "split": split,
                "category": cat,
                "page": int(r.get("page") or r.get("page_start") or 35),
                "doc_type": "RHP",
                "text": text,
            }
        )

    groups = sorted(by_ipo_cat.items(), key=lambda kv: kv[0])
    rng.shuffle(groups)
    out_rows: list[dict[str, Any]] = []
    seen_prompt_hashes: set[tuple[str, str, str]] = set()

    for round_idx in range(6):
        if len(out_rows) >= target_count:
            break
        for (ipo_id, cat), passages_pool in groups:
            if len(out_rows) >= target_count:
                break
            if len(passages_pool) < 2:
                continue
            k = min(len(passages_pool), 2 if round_idx % 2 == 0 else 3)
            sampled_passages = rng.sample(passages_pool, k)
            combined_low = " ".join(p["text"] for p in sampled_passages).lower()
            cand_templates = NEAR_MISS_TEMPLATES.get(cat, []) + NEAR_MISS_TEMPLATES["default"]
            rng.shuffle(cand_templates)

            chosen = None
            for q_en, q_hi, forbidden in cand_templates:
                if any(tok in combined_low for tok in forbidden):
                    continue
                sig = (ipo_id, q_en, sampled_passages[0]["risk_id"])
                if sig not in seen_prompt_hashes:
                    chosen = (q_en, q_hi, forbidden)
                    seen_prompt_hashes.add(sig)
                    break
            if chosen is None:
                continue

            use_hi = rng.random() < hi_ratio
            lang = "hi" if use_hi else "en"
            question = chosen[1] if use_hi else chosen[0]
            abstain = ABSTAIN_HI if use_hi else ABSTAIN_EN
            prompt = render_chat_prompt(question=question, passages=sampled_passages, lang=lang)
            out_rows.append(
                {
                    "id": f"chat_unans_scripted_{len(out_rows) + 1:05d}",
                    "ipo_id": ipo_id,
                    "company": sampled_passages[0]["company"],
                    "split": sampled_passages[0]["split"],
                    "label_source": "rule",
                    "task_type": "chat_unanswerable_scripted",
                    "lang": lang,
                    "question": question,
                    "passages": [format_passage_blocks(sampled_passages)],
                    "answer": abstain,
                    "messages": [
                        {"role": "user", "content": prompt},
                        {"role": "assistant", "content": abstain},
                    ],
                }
            )
    return out_rows


excluded_obj = json.loads((PACK / "data" / "excluded_ipos.json").read_text(encoding="utf-8"))
excluded_ipos = set(excluded_obj["ipos"].keys())

splits_obj = json.loads((PACK / "data" / "splits_by_ipo.json").read_text(encoding="utf-8"))
ipo_to_split = {
    item["ipo_id"]: item["slice"]
    for item in splits_obj["ipos"]
    if not item.get("excluded") and item["ipo_id"] not in excluded_ipos and item["slice"] in ("train", "dev")
}


def load_gz_jsonl(path: pathlib.Path) -> list[dict]:
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


risk_rows: list[dict] = []
for fname, default_split in [("classifier_train.jsonl.gz", "train"), ("classifier_dev.jsonl.gz", "dev")]:
    for r in load_gz_jsonl(PACK / "data" / fname):
        if r["ipo_id"] not in excluded_ipos and r["ipo_id"] in ipo_to_split:
            r["split"] = ipo_to_split[r["ipo_id"]]
            risk_rows.append(r)

for r in risk_rows:
    assert r["ipo_id"] not in excluded_ipos, f"Excluded IPO leaked: {r['ipo_id']}"

target_scripted = 16 if DEBUG else 1500
scripted_rows = build_scripted_near_miss_unanswerables(
    risk_rows=risk_rows,
    excluded_ipos=excluded_ipos,
    target_count=target_scripted,
    seed=SEEDS[0],
)

scripted_path = CKPT_DIR / "scripted_near_miss_unanswerables.jsonl"
with scripted_path.open("w", encoding="utf-8", newline="\n") as f:
    for r in scripted_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Built {len(scripted_rows)} scripted near-miss unanswerables -> {scripted_path}")

In [ ]:
# Cell 3: Teacher generation (~1,150 raw answerable numeric Q&A + ~500 adversarial near-miss unanswerables)
# Checkpoints every chunk (`epoch-<k>`) to CKPT_DIR so RESUME=True continues seamlessly.
import json
import os
import pathlib
import random
import re
import subprocess
import sys
from collections import defaultdict
from typing import Any

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# In DEBUG mode on T4, use the fast 4B model for smoke testing; on G4 (DEBUG=False), use the teacher model.
TEACHER_MODEL = (
    "Qwen/Qwen3-4B-Instruct-2507"
    if DEBUG
    else os.environ.get("FINSIGHT_TEACHER_MODEL", "Qwen/Qwen2.5-32B-Instruct")
)
TARGET_ANSWERABLE_RAW = 8 if DEBUG else 1150  # ~1,150 raw -> >=1,000 kept after numkeep.py
TARGET_ADV_UNANS = 4 if DEBUG else 500
CHUNK_SIZE = 12 if DEBUG else 50

_NUM_RE = re.compile(r"(?:₹|rs\.?|inr|\d+(?:\.\d+)?\s*(?:%|million|crore|lakh))", re.I)

# Build passage bundles (2 multi-number passages from the same non-excluded IPO & category)
by_ipo_cat: dict[tuple[str, str], list[dict]] = defaultdict(list)
for r in risk_rows:
    txt = " ".join(str(r.get("text") or "").split())
    if len(txt.split()) >= 30 and _NUM_RE.search(txt):
        cat = str(r.get("label") or r.get("category") or "financial")
        by_ipo_cat[(r["ipo_id"], cat)].append(
            {
                "risk_id": r.get("risk_id") or r["ipo_id"],
                "ipo_id": r["ipo_id"],
                "company": r.get("company") or r["ipo_id"],
                "split": r["split"],
                "page": int(r.get("page") or 45),
                "doc_type": "RHP",
                "text": txt,
            }
        )

rng = random.Random(SEEDS[1])
passage_bundles: list[dict] = []
for (ipo_id, cat), plist in sorted(by_ipo_cat.items()):
    if len(plist) < 2:
        continue
    rng.shuffle(plist)
    for i in range(0, len(plist) - 1, 2):
        chunk_p = plist[i : i + 2]
        passage_bundles.append(
            {
                "ipo_id": ipo_id,
                "company": chunk_p[0]["company"],
                "split": chunk_p[0]["split"],
                "category": cat,
                "passage_objs": chunk_p,
            }
        )
rng.shuffle(passage_bundles)

total_needed = TARGET_ANSWERABLE_RAW + TARGET_ADV_UNANS
selected_bundles = (passage_bundles * ((total_needed // max(1, len(passage_bundles))) + 1))[:total_needed]

# Resume from existing epoch-<k> checkpoints if present
generated_raw: list[dict] = []
completed_chunks = 0
if RESUME:
    for ckpt in sorted(CKPT_DIR.glob("epoch-*"), key=lambda p: int(p.name.split("-")[1])):
        part_file = ckpt / "teacher_raw_chunk.jsonl"
        if part_file.exists():
            with part_file.open("r", encoding="utf-8") as f:
                generated_raw.extend(json.loads(line) for line in f if line.strip())
            completed_chunks = int(ckpt.name.split("-")[1])
    if generated_raw:
        print(f"Resumed {len(generated_raw)} teacher rows from {completed_chunks} completed chunks.")


def _load_teacher_tokenizer_and_model(model_id: str):
    """Load public or gated HF models safely (tries token=None first so repo-scoped fine-grained tokens never 403)."""
    compute_dtype = (
        torch.bfloat16
        if (torch.cuda.is_available() and torch.cuda.is_bf16_supported())
        else torch.float16
    )
    tok = None
    last_err = None
    for tok_arg in (None, hf_token()):
        try:
            tok = AutoTokenizer.from_pretrained(model_id, token=tok_arg)
            break
        except Exception as exc:
            last_err = exc
    if tok is None:
        raise RuntimeError(f"Failed to load tokenizer for {model_id}: {last_err}")

    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token_id = tok.eos_token_id

    quant_cfg = None
    if not DEBUG:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
            from transformers import BitsAndBytesConfig

            quant_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=compute_dtype,
                bnb_4bit_quant_type="nf4",
            )
        except Exception as exc:
            print("4-bit quantization fallback to fp16/bf16:", exc)

    model = None
    for tok_arg in (None, hf_token()):
        for dtype_key in ("dtype", "torch_dtype"):
            for use_device_map in (True, False):
                kwargs: dict[str, Any] = {dtype_key: compute_dtype}
                if tok_arg is not None:
                    kwargs["token"] = tok_arg
                if use_device_map:
                    kwargs["device_map"] = "auto"
                if quant_cfg is not None:
                    kwargs["quantization_config"] = quant_cfg
                try:
                    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
                    if not use_device_map and quant_cfg is None:
                        model = model.to("cuda" if torch.cuda.is_available() else "cpu")
                    model.eval()
                    return tok, model
                except TypeError as exc:
                    last_err = exc
                    continue
                except Exception as exc:
                    last_err = exc
                    break
    raise RuntimeError(f"Failed to load model {model_id}: {last_err}")


if len(generated_raw) < total_needed:
    print(f"Loading teacher model: {TEACHER_MODEL}")
    tok, teacher_model = _load_teacher_tokenizer_and_model(TEACHER_MODEL)
    model_device = next(teacher_model.parameters()).device

    def generate_teacher_json(system_instruction: str, blocks_text: str) -> dict | None:
        user_msg = f"{system_instruction}\n\nPassages:\n<<<DATA\n{blocks_text}\nDATA>>>\n\nJSON:"
        messages = [{"role": "user", "content": user_msg}]
        try:
            prompt_text = tok.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            prompt_text = tok.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        model_inputs = tok(prompt_text, return_tensors="pt")
        model_inputs = {k: v.to(model_device) for k, v in model_inputs.items() if isinstance(v, torch.Tensor)}
        input_len = model_inputs["input_ids"].shape[1]
        with torch.no_grad():
            out_ids = teacher_model.generate(
                **model_inputs,
                max_new_tokens=220,
                do_sample=False,
                pad_token_id=tok.pad_token_id or tok.eos_token_id,
            )
        gen_text = tok.decode(out_ids[0][input_len:], skip_special_tokens=True).strip()
        # Strip optional <think>...</think> or markdown fences if present
        gen_text = re.sub(r"<think>.*?</think>", "", gen_text, flags=re.S).strip()
        m = re.search(r"\{[^{}]*\}", gen_text, re.S) or re.search(r"\{.*\}", gen_text, re.S)
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except Exception:
            return None

    ANS_INSTRUCTION = (
        "You are generating grounded Q&A training data over Indian IPO RHP passages. "
        "Write ONE specific financial or operational question answered by the passages, and a 1-2 sentence answer "
        "that copies every number AND its unit verbatim (e.g. '₹ 4,720.00 million' or '14.25%') and cites the passage "
        "like [1] or [2] at the end of every sentence. Return ONLY valid JSON: {\"question\": \"...\", \"answer\": \"...\"}."
    )
    UNANS_INSTRUCTION = (
        "You are generating hard near-miss unanswerable questions over Indian IPO RHP passages. "
        "Write ONE plausible investor question on the same topic as the passages, but asking for a specific metric, "
        "fiscal year, or counterparty that is NOT mentioned anywhere in the passages. "
        "Return ONLY valid JSON: {\"question\": \"...\"}."
    )

    start_idx = len(generated_raw)
    chunk_num = completed_chunks
    for batch_start in range(start_idx, total_needed, CHUNK_SIZE):
        chunk_num += 1
        chunk_rows: list[dict] = []
        for idx in range(batch_start, min(batch_start + CHUNK_SIZE, total_needed)):
            b = selected_bundles[idx]
            blocks_str = format_passage_blocks(b["passage_objs"])
            is_ans = idx < TARGET_ANSWERABLE_RAW
            lang = "hi" if (idx % 5 == 0) else "en"
            if is_ans:
                parsed = generate_teacher_json(ANS_INSTRUCTION, blocks_str)
                if parsed and parsed.get("question") and parsed.get("answer"):
                    chunk_rows.append(
                        {
                            "id": f"chat_ans_teacher_{idx + 1:05d}",
                            "ipo_id": b["ipo_id"],
                            "company": b["company"],
                            "split": b["split"],
                            "task_type": "chat_answerable_teacher",
                            "lang": "en",
                            "question": str(parsed["question"]).strip(),
                            "answer": str(parsed["answer"]).strip(),
                            "passage_objs": b["passage_objs"],
                            "passages": [blocks_str],
                        }
                    )
            else:
                parsed = generate_teacher_json(UNANS_INSTRUCTION, blocks_str)
                if parsed and parsed.get("question"):
                    abstain_target = ABSTAIN_HI if lang == "hi" else ABSTAIN_EN
                    chunk_rows.append(
                        {
                            "id": f"chat_unans_teacher_{idx + 1:05d}",
                            "ipo_id": b["ipo_id"],
                            "company": b["company"],
                            "split": b["split"],
                            "task_type": "chat_unanswerable_teacher",
                            "lang": lang,
                            "question": str(parsed["question"]).strip(),
                            "answer": abstain_target,
                            "passage_objs": b["passage_objs"],
                            "passages": [blocks_str],
                        }
                    )

        ckpt_folder = CKPT_DIR / f"epoch-{chunk_num}"
        ckpt_folder.mkdir(parents=True, exist_ok=True)
        with (ckpt_folder / "teacher_raw_chunk.jsonl").open("w", encoding="utf-8", newline="\n") as f:
            for r in chunk_rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        generated_raw.extend(chunk_rows)
        print(f"[Chunk epoch-{chunk_num}] Saved {len(chunk_rows)} rows (total={len(generated_raw)}/{total_needed})")

In [ ]:
# Cell 4: Run ml/handoff/eval/numkeep.py + citation/length filter, mix with student_v1_{train,dev}.jsonl.gz, save SFT sets
import datetime


def filter_teacher_chat_rows(
    raw_rows: list[dict[str, Any]],
    excluded_ipos: set[str],
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, Any]]:
    kept: list[dict[str, Any]] = []
    rejected: list[dict[str, Any]] = []
    reasons: Counter[str] = Counter()

    for row in raw_rows:
        ipo_id = str(row.get("ipo_id") or "").strip()
        if not ipo_id or ipo_id in excluded_ipos or row.get("excluded"):
            reasons["excluded_ipo"] += 1
            rejected.append({**row, "reject_reason": "excluded_ipo"})
            continue
        split = str(row.get("split") or row.get("slice") or "train").strip()
        if split not in ("train", "dev"):
            reasons["invalid_split"] += 1
            rejected.append({**row, "reject_reason": "invalid_split"})
            continue

        lang = str(row.get("lang") or "en").strip()
        expected_abstain = ABSTAIN_HI if lang == "hi" else ABSTAIN_EN
        task_type = str(row.get("task_type") or "chat_answerable_teacher").strip()
        answer = str(row.get("answer") or "").strip()
        question = str(row.get("question") or "").strip()
        passage_objs = row.get("passage_objs") or [
            {"page": 1, "doc_type": "RHP", "text": t} for t in (row.get("passages") or [])
        ]
        passages_text = format_passage_blocks(passage_objs)

        if not question or not answer or not passages_text.strip():
            reasons["empty_field"] += 1
            rejected.append({**row, "reject_reason": "empty_field"})
            continue

        if task_type == "chat_unanswerable_teacher":
            if answer not in (ABSTAIN_EN, ABSTAIN_HI):
                answer = expected_abstain
            missing_nums = unmatched_numbers(passages_text, answer)
            if missing_nums:
                reasons["unmatched_numbers"] += 1
                rejected.append({**row, "reject_reason": "unmatched_numbers", "unmatched_numbers": missing_nums})
                continue
        else:
            if answer in (ABSTAIN_EN, ABSTAIN_HI) or "could not find this in the document" in answer.lower():
                reasons["unexpected_abstain"] += 1
                rejected.append({**row, "reject_reason": "unexpected_abstain"})
                continue
            if not _CITATION_RE.search(answer):
                reasons["missing_citation"] += 1
                rejected.append({**row, "reject_reason": "missing_citation"})
                continue
            if len(answer.split()) > 120:
                reasons["too_long_over_120_words"] += 1
                rejected.append({**row, "reject_reason": "too_long_over_120_words"})
                continue
            missing_nums = unmatched_numbers(passages_text, answer)
            if missing_nums:
                reasons["unmatched_numbers"] += 1
                rejected.append({**row, "reject_reason": "unmatched_numbers", "unmatched_numbers": missing_nums})
                continue

        prompt = render_chat_prompt(question=question, passages=passage_objs, lang=lang)
        kept.append(
            {
                "id": row.get("id") or f"{task_type}_{len(kept) + 1:05d}",
                "ipo_id": ipo_id,
                "company": row.get("company") or ipo_id,
                "split": split,
                "label_source": "ai",
                "task_type": task_type,
                "lang": lang,
                "question": question,
                "passages": [passages_text],
                "answer": answer,
                "messages": [
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": answer},
                ],
            }
        )

    total = len(kept) + len(rejected)
    stats = {
        "kept": len(kept),
        "dropped": len(rejected),
        "drop_rate": round(len(rejected) / max(1, total), 4),
        "reasons": dict(reasons),
    }
    return kept, rejected, stats


def mix_rewrite_and_chat_sft(
    rewrite_train: list[dict[str, Any]],
    rewrite_dev: list[dict[str, Any]],
    chat_rows: list[dict[str, Any]],
    excluded_ipos: set[str],
    seed: int = 42,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, Any]]:
    rng = random.Random(seed)
    clean_rw_train = [r for r in rewrite_train if r.get("ipo_id") not in excluded_ipos]
    clean_rw_dev = [r for r in rewrite_dev if r.get("ipo_id") not in excluded_ipos]
    chat_train = [r for r in chat_rows if r.get("split") == "train" and r.get("ipo_id") not in excluded_ipos]
    chat_dev = [r for r in chat_rows if r.get("split") == "dev" and r.get("ipo_id") not in excluded_ipos]
    combined_train = list(clean_rw_train) + list(chat_train)
    combined_dev = list(clean_rw_dev) + list(chat_dev)
    rng.shuffle(combined_train)
    rng.shuffle(combined_dev)
    task_counts = Counter(r.get("task_type", "risk_rewrite") for r in chat_train + chat_dev)
    source_counts = Counter(r.get("label_source", "ai") for r in combined_train + combined_dev)
    summary = {
        "rewrite_train": len(clean_rw_train),
        "rewrite_dev": len(clean_rw_dev),
        "chat_train": len(chat_train),
        "chat_dev": len(chat_dev),
        "total_train": len(combined_train),
        "total_dev": len(combined_dev),
        "chat_breakdown": dict(task_counts),
        "label_source_mix": {
            "human": int(source_counts.get("human", 0)),
            "human_verified_ai": int(source_counts.get("human_verified_ai", 0)),
            "ai": int(source_counts.get("ai", 0)),
            "rule": int(source_counts.get("rule", 0)),
        },
    }
    return combined_train, combined_dev, summary


raw_teacher_path = CKPT_DIR / "teacher_raw.jsonl"
with raw_teacher_path.open("w", encoding="utf-8", newline="\n") as f:
    for r in generated_raw:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# 1. Run CC's exact ml/handoff/eval/numkeep.py script on teacher_raw.jsonl
numkeep_out_path = CKPT_DIR / "teacher_numkeep_only.jsonl"
numkeep_proc = subprocess.run(
    [
        sys.executable,
        str(PACK / "eval" / "numkeep.py"),
        "--in",
        str(raw_teacher_path),
        "--out",
        str(numkeep_out_path),
        "--source-field",
        "passages",
        "--output-field",
        "answer",
    ],
    capture_output=True,
    text=True,
    check=True,
)
numkeep_cli_stats = json.loads(numkeep_proc.stdout.strip())
print("numkeep.py CLI stats:", numkeep_cli_stats)

# 2. Apply full filter_teacher_chat_rows (numkeep + [n] citation + <=120 words + frozen chat prompt rendering)
kept_teacher, rejected_teacher, full_filter_stats = filter_teacher_chat_rows(
    raw_rows=generated_raw,
    excluded_ipos=excluded_ipos,
)

kept_path = CKPT_DIR / "teacher_chat_kept.jsonl"
rej_path = CKPT_DIR / "teacher_chat_kept.rejected.jsonl"
with kept_path.open("w", encoding="utf-8", newline="\n") as f:
    for r in kept_teacher:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
with rej_path.open("w", encoding="utf-8", newline="\n") as f:
    for r in rejected_teacher:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# 3. Mix with existing rewrite pairs (student_v1_train.jsonl.gz 10,362 / student_v1_dev.jsonl.gz 529)
rw_train = load_gz_jsonl(PACK / "data" / "student_v1_train.jsonl.gz")
rw_dev = load_gz_jsonl(PACK / "data" / "student_v1_dev.jsonl.gz")
if DEBUG:
    rw_train = rw_train[:32]
    rw_dev = rw_dev[:16]

all_chat_rows = scripted_rows + kept_teacher
combined_train, combined_dev, mix_summary = mix_rewrite_and_chat_sft(
    rewrite_train=rw_train,
    rewrite_dev=rw_dev,
    chat_rows=all_chat_rows,
    excluded_ipos=excluded_ipos,
    seed=SEEDS[0],
)

train_out = CKPT_DIR / "student_v2_sft_train.jsonl.gz"
dev_out = CKPT_DIR / "student_v2_sft_dev.jsonl.gz"
with gzip.open(train_out, "wt", encoding="utf-8", newline="\n") as f:
    for r in combined_train:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
with gzip.open(dev_out, "wt", encoding="utf-8", newline="\n") as f:
    for r in combined_dev:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

wall_minutes = round((time.time() - t0_wall) / 60.0, 2)
hw_name = "T4-DEBUG" if DEBUG else "G4"
units_rate = 1.60 if DEBUG else 8.71
units_used = round((wall_minutes / 60.0) * units_rate, 2)

gen_report = {
    "job": JOB,
    "debug": DEBUG,
    "teacher_model": TEACHER_MODEL,
    "numkeep_cli_stats": numkeep_cli_stats,
    "full_filter_stats": full_filter_stats,
    "mix_summary": mix_summary,
    "compute": {
        "hardware": hw_name,
        "units_used": units_used,
        "wall_minutes": wall_minutes,
    },
    "date": datetime.date.today().isoformat(),
}
(CKPT_DIR / "generation_stats.json").write_text(json.dumps(gen_report, indent=2), encoding="utf-8")
print(json.dumps(gen_report, indent=2))